In [384]:
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Annotated, List

import random

import numpy as np
from numpy.lib.stride_tricks import sliding_window_view

from geneticengine.grammar.metahandlers.ints import IntRange
from geneticengine.grammar import extract_grammar
from geneticengine.grammar.decorators import weight, abstract
from geneticengine.problems import SingleObjectiveProblem, MultiObjectiveProblem
from geneticengine.random.sources import NativeRandomSource
from geneticengine.algorithms.gp.gp import GeneticProgramming
from geneticengine.evaluation.budget import TimeBudget, EvaluationBudget
from geneticengine.representations.tree.initializations import MaxDepthDecider, FullDecider, ProgressivelyTerminalDecider, PositionIndependentGrowDecider
from geneticengine.representations.tree.operators import GrowInitializer, PositionIndependentGrowInitializer, FullInitializer, RampedHalfAndHalfInitializer
from geneticengine.algorithms.gp.operators.initializers import HalfAndHalfInitializer, StandardInitializer
from geneticengine.representations.tree.treebased import TreeBasedRepresentation
from geneticengine.representations.grammatical_evolution.structured_ge import StructuredGrammaticalEvolutionRepresentation
from geneticengine.evaluation.recorder import CSVSearchRecorder
from geneticengine.evaluation.tracker import ProgressTracker
from geneticengine.evaluation.parallel import ParallelEvaluator

from geneticengine.algorithms.gp.operators.combinators import ParallelStep, SequenceStep
from geneticengine.algorithms.gp.operators.crossover import GenericCrossoverStep
from geneticengine.algorithms.gp.operators.elitism import ElitismStep
from geneticengine.algorithms.gp.operators.mutation import GenericMutationStep
from geneticengine.algorithms.gp.operators.novelty import NoveltyStep
from geneticengine.algorithms.gp.operators.selection import LexicaseSelection, TournamentSelection

from geneticengine.solutions.individual import Individual, PhenotypicIndividual
from geneticengine.algorithms.gp.structure import GeneticStep
from geneticengine.problems import Problem
from geneticengine.random.sources import RandomSource
from geneticengine.representations.api import RepresentationWithCrossover, Representation
from geneticengine.evaluation import Evaluator
from typing import Iterator, Any, TypeVar

from sklearn.datasets import load_breast_cancer

import pandas as pd

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import f1_score, confusion_matrix, classification_report, ConfusionMatrixDisplay, roc_auc_score, roc_curve, auc
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE

import time

import seaborn as sns
import matplotlib.pyplot as plt

import os     

from functools import lru_cache

In [385]:
# model_used = DecisionTreeClassifier(random_state=42, max_depth=6,max_features='log2', min_samples_leaf=5, min_samples_split=10)
# model_used = LogisticRegression(random_state=42)
model_used = RandomForestClassifier(random_state=42, n_estimators=20, n_jobs=-1)
target_fpr_value = 0.05

### Functions and Data Preprocessing

In [386]:
def plot_roc(fpr, tpr):
    plt.plot(fpr, tpr, label='ROC curve')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC curve')
    plt.legend()
    plt.show()

def evaluate(predictions, FIXED_FPR = target_fpr_value):
    fprs, tprs, thresholds = roc_curve(y_test, predictions)
    plot_roc(fprs, tprs)
    tpr = tprs[fprs<FIXED_FPR][-1]
    fpr = fprs[fprs<FIXED_FPR][-1]
    threshold = thresholds[fprs<FIXED_FPR][-1]
        
    print("AUC:", roc_auc_score(y_test, predictions))
    to_pct = lambda x: str(round(x, 4) * 100) + "%"
    print("TPR: ", to_pct(tpr), "\nFPR: ", to_pct(fpr), "\nThreshold: ", round(threshold, 2))

In [387]:
df = pd.read_csv('base.csv')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 32 columns):
 #   Column                            Non-Null Count    Dtype  
---  ------                            --------------    -----  
 0   fraud_bool                        1000000 non-null  int64  
 1   income                            1000000 non-null  float64
 2   name_email_similarity             1000000 non-null  float64
 3   prev_address_months_count         1000000 non-null  int64  
 4   current_address_months_count      1000000 non-null  int64  
 5   customer_age                      1000000 non-null  int64  
 6   days_since_request                1000000 non-null  float64
 7   intended_balcon_amount            1000000 non-null  float64
 8   payment_type                      1000000 non-null  object 
 9   zip_count_4w                      1000000 non-null  int64  
 10  velocity_6h                       1000000 non-null  float64
 11  velocity_24h                      1000

In [388]:
df.head(2)

,fraud_bool,income,name_email_similarity,prev_address_months_count,current_address_months_count,customer_age,days_since_request,intended_balcon_amount,payment_type,zip_count_4w,...,has_other_cards,proposed_credit_limit,foreign_request,source,session_length_in_minutes,device_os,keep_alive_session,device_distinct_emails_8w,device_fraud_count,month
0,0,0.3,0.986506,-1,25,40,0.006735,102.453711,AA,1059,...,0,1500.0,0,INTERNET,16.224843,linux,1,1,0,0
1,0,0.8,0.617426,-1,89,20,0.010095,-0.849551,AD,1658,...,0,1500.0,0,INTERNET,3.363854,other,1,1,0,0


In [389]:
categorical_features = [col for col in df.columns if df[col].dtype == 'object']

print(categorical_features)

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

encoded_data = encoder.fit_transform(df[categorical_features])

encoded_df = pd.DataFrame(encoded_data, columns=encoder.get_feature_names_out(categorical_features))

df = df.drop(columns=categorical_features).reset_index(drop=True)
df = pd.concat([df, encoded_df], axis=1)

df.head(2)

['payment_type', 'employment_status', 'housing_status', 'source', 'device_os']


,fraud_bool,income,name_email_similarity,prev_address_months_count,current_address_months_count,customer_age,days_since_request,intended_balcon_amount,zip_count_4w,velocity_6h,...,housing_status_BE,housing_status_BF,housing_status_BG,source_INTERNET,source_TELEAPP,device_os_linux,device_os_macintosh,device_os_other,device_os_windows,device_os_x11
0,0,0.3,0.986506,-1,25,40,0.006735,102.453711,1059,13096.035018,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
1,0,0.8,0.617426,-1,89,20,0.010095,-0.849551,1658,9223.283431,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0


In [390]:
df['month'].value_counts().sort_index()

month
0    132440
1    127620
2    136979
3    150936
4    127691
5    119323
6    108168
7     96843
Name: count, dtype: int64

In [391]:
#get the first 50% of the dataset
split_point = int(len(df) * 0.3)
X = df.drop(['fraud_bool'], axis=1)
y = df['fraud_bool']

X = X[:split_point]
y = y[:split_point]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False)

# X_train = X[X['month']<3]
# X_test = X[X['month']==3]
# y_train = y[X['month']<3]
# y_test = y[X['month']==3]

X_train.drop('month', axis=1, inplace=True)
X_test.drop('month', axis=1, inplace=True)

In [392]:
print(y_train.value_counts(),y_test.value_counts())

fraud_bool
0    237479
1      2521
Name: count, dtype: int64 fraud_bool
0    59493
1      507
Name: count, dtype: int64


In [393]:
# features_to_scale = [col for col in X_train.select_dtypes(include=np.number).columns.tolist() if col not in encoded_df.columns]

# scaler = StandardScaler()
# X_train[features_to_scale] = scaler.fit_transform(X_train[features_to_scale])
# X_test[features_to_scale] = scaler.transform(X_test[features_to_scale])

### Baseline Model

In [394]:
feature_names = X_train.columns.tolist()
n_features = len(feature_names)

In [395]:
model = model_used

model.fit(X_train, y_train)

train_probs = model.predict_proba(X_train)[:,1]

fpr, tpr, thresholds = roc_curve(y_train, train_probs)

target_fpr = target_fpr_value

if np.any(fpr <= target_fpr):
    valid_indices = np.where(fpr <= target_fpr)[0] #where returns a tuple
    best_index = valid_indices[np.argmax(tpr[valid_indices])]
    train_tpr_at_fpr = tpr[best_index]

probs = model.predict_proba(X_test)[:,1]

fpr, tpr, thresholds = roc_curve(y_test, probs)


baseline_recall_at_fpr = 0.0

if np.any(fpr <= target_fpr):
    valid_indices = np.where(fpr <= target_fpr)[0] #where returns a tuple
    best_index = valid_indices[np.argmax(tpr[valid_indices])]
    baseline_tpr_at_fpr = tpr[best_index]

# plt.figure(figsize=(20,10))
# plot_tree(model, feature_names=feature_names, class_names=['No Fraud', 'Fraud'], filled=True)
# plt.savefig("tree_based.svg", format='svg')

print(f"Train TPR: {train_tpr_at_fpr}, Test TPR: {baseline_tpr_at_fpr}")

Train TPR: 1.0, Test TPR: 0.34122287968441817


### Feature Selection with and model performance

#### Feature Importance

In [396]:
# feature_importances = model.feature_importances_
# importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': feature_importances}).sort_values(by='Importance', ascending=False)

# print(importance_df)

In [397]:
# importances = model.feature_importances_     
# labels = np.array(feature_names)             
                                             
# order = np.argsort(importances)[::-1]        
# sorted_importances = importances[order]      
# sorted_labels = labels[order]                

# plt.figure(figsize=(15, 6))                                
# plt.plot(range(1, len(sorted_importances) +  
# 1), sorted_importances, marker="o")          
# plt.xticks(range(1, len(sorted_labels) + 1), 
# sorted_labels, rotation=45, ha="right")      
# plt.ylabel("Importance")                     
# plt.xlabel("Ranked Features")                
# plt.tight_layout()                           
# plt.show()  

In [398]:
    # selected_features = importance_df.loc[importance_df['Importance'] >= 0.01, 'Feature']
    # X_train = X_train[selected_features]
    # X_test = X_test[selected_features]

    # feature_names = X_train.columns.tolist()
    # n_features = len(feature_names)

    # n_features

In [399]:
# corr_matrix = df.corr(method='pearson').abs()

# upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# to_drop = [column for column in upper_tri.columns if any(upper_tri[column] > 0.9)]

# print(to_drop)


In [400]:
# importances = model.feature_importances_     
# labels = np.array(feature_names)             
                                             
# order = np.argsort(importances)[::-1]        
# sorted_importances = importances[order]      
# sorted_labels = labels[order]                

# plt.figure(figsize=(15, 6))                                
# plt.plot(range(1, len(sorted_importances) +  
# 1), sorted_importances, marker="o")          
# plt.xticks(range(1, len(sorted_labels) + 1), 
# sorted_labels, rotation=45, ha="right")      
# plt.ylabel("Importance")                     
# plt.xlabel("Ranked Features")                
# plt.tight_layout()                           
# plt.show()  

In [401]:
# selected_features = importance_df.loc[importance_df['Importance']>0, 'Feature']
# X_train = X_train[selected_features]
# X_test = X_test[selected_features]
# # X_train.shape[1]

# feature_names = X_train.columns.tolist()
# n_features = len(feature_names)

# n_features


#### RFE

In [402]:
# selector = RFE(model, n_features_to_select=0.8, step=1)
# selector.fit(X_train, y_train)

In [403]:
# ranking = selector.ranking_
# selected_features = [feature for feature, rank in zip(feature_names, ranking) if rank == 1]

# X_train = X_train[selected_features]
# X_test = X_test[selected_features]

# feature_names = X_train.columns.tolist()
# n_features = len(feature_names)

# n_features

#### Model Baseline

In [404]:
# model = model_used

# model.fit(X_train, y_train)

# probs = model.predict_proba(X_test)[:,1]

# fpr, tpr, thresholds = roc_curve(y_test, probs)

# target_fpr = target_fpr_value
# baseline_recall_at_fpr = 0.0

# if np.any(fpr <= target_fpr):
#     valid_indices = np.where(fpr <= target_fpr)[0] #where returns a tuple
#     best_index = valid_indices[np.argmax(tpr[valid_indices])]
#     baseline_tpr_at_fpr = tpr[best_index]

# print(f"TPR: {baseline_tpr_at_fpr}")

### Grammar

In [405]:
# X_np = X_train.to_numpy()
# lookback = 5
# feature_col = X_np[:, 3]
# feature_col_rev = feature_col[::-1]
# vector = sliding_window_view(feature_col_rev, window_shape=lookback)[::-1, ::-1]
# initial_rows = np.array([feature_col[:i] for i in range(0, lookback)], dtype=object)
# initial_rows = np.array([np.concatenate((np.full(lookback-len(row), np.nan), row)) for row in initial_rows])
# vector = np.vstack((initial_rows, vector))

In [406]:
@dataclass
class Value(ABC):
    def evaluate(self):
        pass

class Scalar(ABC):
    pass

class Vectorial(ABC):
    pass


In [407]:
@weight(0.1)
@abstract
class InternalScalar(Scalar):
    pass

@weight(0.9)
@abstract
class RootScalar(Scalar):
    pass

In [408]:
@weight(0.5)
@dataclass #Scalar Features (1)
class ScalarVar(RootScalar): 
    index: Annotated[int, IntRange(0,n_features-1)]

    def evaluate(self, X_np):
        return X_np[:, self.index]
    
    def __str__(self):
        return feature_names[self.index]

# @weight(0.05) 
@dataclass #Vectorial Features ([1,2,3,4])
class VectorialVar(Vectorial):
    index: Annotated[int, IntRange(0, n_features-1)]
    lookback: Annotated[int, IntRange(5,15)]

    def evaluate(self, X_np):
        feature_col = X_np[:, self.index]
        feature_col_rev = feature_col[::-1]
        vector = sliding_window_view(feature_col_rev, window_shape=self.lookback)[::-1, ::-1]
        initial_rows = np.array([feature_col[:i] for i in range(0, self.lookback)], dtype=object)
        initial_rows = np.array([np.concatenate((np.full(self.lookback-len(row), np.nan), row)) for row in initial_rows])
        vector = np.vstack((initial_rows, vector))
        return vector

In [409]:
#scalar -> scalar
@weight(0.2)
@dataclass 
class Add(RootScalar):
    right: Scalar
    left: Scalar

    def evaluate(self, X_np):
        return self.left.evaluate(X_np) + self.right.evaluate(X_np)
    
    def __str__(self):
        return f"({self.left} + {self.right})"

@weight(0.2)
@dataclass
class Subtract(RootScalar):
    right: Scalar
    left: Scalar

    def evaluate(self, X_np):
        return (self.left.evaluate(X_np)) - (self.right.evaluate(X_np))
    
    def __str__(self):
        return f"({self.left} - {self.right})"

@weight(0.1)
@dataclass
class Multiply(RootScalar):
    right: Scalar
    left: Scalar

    def evaluate(self, X_np):
        return self.left.evaluate(X_np) * self.right.evaluate(X_np)
    
    def __str__(self):
        return f"({self.left} * {self.right})"

@weight(0.33)
@dataclass
class Mean_Vectorial(InternalScalar):
    arr: Vectorial

    def evaluate(self, X_np):
        return np.nanmean(self.arr.evaluate(X_np))
    
    def __str__(self):
        return f"Mean({self.arr})"
    
@weight(0.33)
@dataclass
class Min_Vectorial(InternalScalar):
    arr: Vectorial

    def evaluate(self, X_np):
        return np.nanmin(self.arr.evaluate(X_np))
    
    def __str__(self):
        return f"Min({self.arr})"
    
@weight(0.33)
@dataclass
class Max_Vectorial(InternalScalar):
    arr: Vectorial

    def evaluate(self, X_np):
        return np.nanmax(self.arr.evaluate(X_np))
    
    def __str__(self):
        return f"Max({self.arr})"



In [410]:
grammar = extract_grammar([Add, Subtract, Multiply,ScalarVar, Mean_Vectorial, Min_Vectorial, Max_Vectorial, VectorialVar], RootScalar)
print(f"Grammar: {repr(grammar)}")
print("-------------------------")
print(f"Grammar: {repr(grammar.usable_grammar())}")

Grammar: Grammar<Starting=RootScalar,Productions={
RootScalar -> Add(right: Scalar, left: Scalar)<0.20>|
	Subtract(right: Scalar, left: Scalar)<0.20>|
	Multiply(right: Scalar, left: Scalar)<0.10>|
	ScalarVar(index: Annotated[int])<0.50>

Scalar -> InternalScalar()<0.10>|
	RootScalar()<0.90>

InternalScalar -> Min_Vectorial(arr: Vectorial)<0.33>|
	Max_Vectorial(arr: Vectorial)<0.33>|
	Mean_Vectorial(arr: Vectorial)<0.33>

Vectorial -> VectorialVar(index: Annotated[int], lookback: Annotated[int])<1.00>
}
-------------------------
Grammar: Grammar<Starting=RootScalar,Productions={
Scalar -> InternalScalar()<0.10>|
	RootScalar()<0.90>

InternalScalar -> Max_Vectorial(arr: Vectorial)<0.33>|
	Mean_Vectorial(arr: Vectorial)<0.33>|
	Min_Vectorial(arr: Vectorial)<0.33>

Vectorial -> VectorialVar(index: Annotated[int], lookback: Annotated[int])<1.00>

RootScalar -> ScalarVar(index: Annotated[int])<0.50>|
	Subtract(right: Scalar, left: Scalar)<0.20>|
	Add(right: Scalar, left: Scalar)<0.20>|
	Mult

### Fitness and GP

In [411]:
ARCHIVE_TRAIN_DF = X_train.copy()
ARCHIVE_TEST_DF = X_test.copy()

ARCHIVE_TEMP : list[Individual] = []

In [412]:
X_train_np = ARCHIVE_TRAIN_DF.to_numpy()
X_test_np = ARCHIVE_TEST_DF.to_numpy()

def fitness_function(individual: Scalar): #individual -> expression
    start = time.perf_counter()
    train_feature = individual.evaluate(X_train_np)
    test_feature = individual.evaluate(X_test_np)
    if train_feature.ndim == 0:
        train_feature = np.full(X_train_np.shape[0], train_feature)
    if test_feature.ndim == 0:
        test_feature = np.full(X_test_np.shape[0], test_feature)

    X_train_augmented = np.c_[X_train_np, np.array(train_feature).reshape(-1,1)]
    X_test_augmented = np.c_[X_test_np, np.array(test_feature).reshape(-1,1)]

    model = model_used
    
    model.fit(X_train_augmented, y_train)
    
    probs = model.predict_proba(X_test_augmented)[:, 1]

    fpr, tpr, thresholds = roc_curve(y_test, probs)

    if np.any(fpr <= target_fpr):
        valid_indices = np.where(fpr<=target_fpr)[0]
        best_indice = valid_indices[np.argmax(tpr[valid_indices])]
        tpr_at_fpr = tpr[best_indice]

    tpr_diff = tpr_at_fpr-baseline_tpr_at_fpr
        
    features, num_operations = analyse_complexity(individual)

    end = time.perf_counter()
    elapsed = end - start
    return [tpr_at_fpr, tpr_diff, num_operations, elapsed]



In [413]:
def analyse_complexity(individual: Scalar):
    if isinstance(individual, ScalarVar) or isinstance(individual, VectorialVar):
        return {individual.index}, 0 #unique feature
    
    total_features = set()
    total_operations = 1
    if hasattr(individual, 'left') and hasattr(individual, 'right'):
        left_features, left_operations = analyse_complexity(individual.left)
        right_features, right_operations = analyse_complexity(individual.right)
        total_features.update(left_features)
        total_features.update(right_features)
        total_operations += left_operations + right_operations
    elif hasattr(individual, 'arr'):
        arr_features, arr_operations = analyse_complexity(individual.arr)
        total_features.update(arr_features)
        total_operations += arr_operations
    return total_features, total_operations


In [414]:
class ArchiveStep(GeneticStep):
    def iterate(
        self,
        problem: Problem,
        evaluator: Evaluator,
        representation: Representation,
        random: RandomSource,
        population: Iterator[PhenotypicIndividual],
        target_size: int,
        generation: int,
    ) -> Iterator[PhenotypicIndividual]:
        global ARCHIVE_TEMP, train_tpr_at_fpr, baseline_tpr_at_fpr, ARCHIVE_TRAIN_DF, ARCHIVE_TEST_DF, X_train_np, X_test_np
        best_fitness = 0
        for i, individual in enumerate(population):
            # if individual.get_fitness(problem).fitness_components[0] > baseline_tpr_at_fpr and individual.get_fitness(problem).fitness_components[0] > best_fitness:
            if individual.get_fitness(problem).fitness_components[0] > baseline_tpr_at_fpr:
                # best_fitness = individual.get_fitness(problem).fitness_components[0]
                print("New Individual:", str(individual.get_phenotype()), "Fitness:", individual.get_fitness(problem).fitness_components)
                # if not ARCHIVE_TEMP:
                ARCHIVE_TEMP.append(individual)
                # else:
                    # ARCHIVE_TEMP[0] = individual
            yield individual

        if ARCHIVE_TEMP:
            print(f"Archive Size: {len(ARCHIVE_TEMP)}")
            for ind in ARCHIVE_TEMP:
                train_feature_new = ind.get_phenotype().evaluate(X_train_np)
                test_feature_new = ind.get_phenotype().evaluate(X_test_np)

                ARCHIVE_TRAIN_DF[str(ind)] = train_feature_new
                ARCHIVE_TEST_DF[str(ind)] = test_feature_new

            ARCHIVE_TRAIN_DF = ARCHIVE_TRAIN_DF.loc[:, ~ARCHIVE_TRAIN_DF.columns.duplicated()]
            ARCHIVE_TEST_DF = ARCHIVE_TEST_DF.loc[:, ~ARCHIVE_TEST_DF.columns.duplicated()]

            ARCHIVE_TRAIN_DF.columns = [str(col) for col in ARCHIVE_TRAIN_DF.columns]
            ARCHIVE_TEST_DF.columns = [str(col) for col in ARCHIVE_TEST_DF.columns]

            model = model_used
            model.fit(ARCHIVE_TRAIN_DF, y_train)
            target_fpr = target_fpr_value
            train_probs = model.predict_proba(ARCHIVE_TRAIN_DF)[:,1]
            fpr, tpr, thresholds = roc_curve(y_train, train_probs)
            if np.any(fpr <= target_fpr):
                valid_indices = np.where(fpr <= target_fpr)[0]
                best_index = valid_indices[np.argmax(tpr[valid_indices])]
                train_tpr_at_fpr = tpr[best_index]
            print(f"Train TPR: {train_tpr_at_fpr}, Train shape: {ARCHIVE_TRAIN_DF.shape}")
            probs = model.predict_proba(ARCHIVE_TEST_DF)[:,1]
            fpr, tpr, thresholds = roc_curve(y_test, probs)
            if np.any(fpr <= target_fpr):
                valid_indices = np.where(fpr <= target_fpr)[0]
                best_index = valid_indices[np.argmax(tpr[valid_indices])]
                baseline_tpr_at_fpr = tpr[best_index]
            print(f"Test TPR: {baseline_tpr_at_fpr}, Test shape: {ARCHIVE_TEST_DF.shape}")
            ARCHIVE_TEMP = []
            X_train_np = ARCHIVE_TRAIN_DF.to_numpy()
            X_test_np = ARCHIVE_TEST_DF.to_numpy()
        

In [415]:
def lexicase_step():
    return SequenceStep(
        ArchiveStep(),
        ParallelStep(
            [
                ElitismStep(),
                NoveltyStep(),
                SequenceStep(
                    LexicaseSelection(epsilon=True),
                    # TournamentSelection(tournament_size=3),
                    GenericCrossoverStep(0.9),
                    GenericMutationStep(0.1),
                )
            ],
            weights=[0.05, 0.05, 0.9]
        ),
    )

prob = MultiObjectiveProblem(
    fitness_function=fitness_function,
    # minimize=[False, False, False, False, True],
    minimize=[False, False, True, True],
)
r = NativeRandomSource(123)
alg = GeneticProgramming(
    problem=prob,
    budget=TimeBudget(1800),
    population_size=25,
    representation=TreeBasedRepresentation(grammar, ProgressivelyTerminalDecider(r, grammar)),
    random=r,
    step=lexicase_step(),
    tracker=ProgressTracker(
        prob,
        recorders=[CSVSearchRecorder(
            csv_path='feedzai_output_pastvalues.csv', 
            problem=prob, 
            fields={
                    "Eval Time": lambda t,i,p: i.get_fitness(p).fitness_components[3],
                    "TPR Test": lambda t,i,p: i.get_fitness(p).fitness_components[0],
                    "TPR Test Diff": lambda t,i,p: i.get_fitness(p).fitness_components[1],
                    "Expression": lambda t, i, p: i.get_phenotype(),
                    "Num Operations": lambda t,i,p: i.get_fitness(p).fitness_components[2],
                    'Generation': lambda t,i,p: i.metadata["generation"]
                    },
            only_record_best_individuals=False)]
    )
    
)

solutions = alg.search()

New Individual: ((Min(VectorialVar(index=2, lookback=9)) + Min(VectorialVar(index=18, lookback=11))) - (Min(VectorialVar(index=28, lookback=6)) + Min(VectorialVar(index=24, lookback=6)))) Fitness: [np.float64(0.3510848126232742), np.float64(0.00986193293885601), 7, 2.3574226000346243]
New Individual: foreign_request Fitness: [np.float64(0.3530571992110454), np.float64(0.011834319526627224), 0, 2.4026931000407785]
New Individual: employment_status_CD Fitness: [np.float64(0.3431952662721893), np.float64(0.0019723865877711577), 0, 2.2996428000042215]
New Individual: prev_address_months_count Fitness: [np.float64(0.34714003944773175), np.float64(0.005917159763313584), 0, 2.366419499972835]
New Individual: ((Min(VectorialVar(index=1, lookback=15)) + Min(VectorialVar(index=2, lookback=13))) + (Min(VectorialVar(index=44, lookback=15)) + Min(VectorialVar(index=27, lookback=5)))) Fitness: [np.float64(0.3510848126232742), np.float64(0.00986193293885601), 7, 2.3489810000173748]
New Individual: ((

In [416]:
#create a copy of ARCHIVE_TRAIN_DF and ARCHIVE_TEST_DF to use later
# ARCHIVE_TRAIN_DF_FINAL = ARCHIVE_TRAIN_DF.copy()
# ARCHIVE_TEST_DF_FINAL = ARCHIVE_TEST_DF.copy()

In [417]:
# #train a model with the final augmented data
# model = model_used
# model.fit(ARCHIVE_TRAIN_DF_FINAL, y_train)
# target_fpr = target_fpr_value
# probs = model.predict_proba(ARCHIVE_TEST_DF_FINAL)[:,1]
# fpr, tpr, thresholds = roc_curve(y_test, probs)
# if np.any(fpr <= target_fpr):
#     valid_indices = np.where(fpr <= target_fpr)[0]
#     best_index = valid_indices[np.argmax(tpr[valid_indices])]
#     final_tpr_at_fpr = tpr[best_index]
# print(f"Final Test TPR: {final_tpr_at_fpr}, Test shape: {ARCHIVE_TEST_DF_FINAL.shape}")

# #remove the last column in both dataasets
# ARCHIVE_TRAIN_DF_FINAL = ARCHIVE_TRAIN_DF_FINAL.iloc[:,:-1]
# ARCHIVE_TEST_DF_FINAL = ARCHIVE_TEST_DF_FINAL.iloc[:,:-1]
# model = model_used
# model.fit(ARCHIVE_TRAIN_DF_FINAL, y_train)
# target_fpr = target_fpr_value
# probs = model.predict_proba(ARCHIVE_TEST_DF_FINAL)[:,1]
# fpr, tpr, thresholds = roc_curve(y_test, probs)
# if np.any(fpr <= target_fpr):
#     valid_indices = np.where(fpr <= target_fpr)[0]
#     best_index = valid_indices[np.argmax(tpr[valid_indices])]
#     final_tpr_at_fpr = tpr[best_index]
# print(f"Final Test TPR: {final_tpr_at_fpr}, Test shape: {ARCHIVE_TEST_DF_FINAL.shape}")


In [418]:
# model = model_used
# model.fit(ARCHIVE_TRAIN_DF, y_train)
# train_probs = model.predict_proba(ARCHIVE_TRAIN_DF)[:,1]
# fpr, tpr, thresholds = roc_curve(y_train, train_probs)
# target_fpr = target_fpr_value
# if np.any(fpr <= target_fpr):
#     valid_indices = np.where(fpr <= target_fpr)[0]
#     best_index = valid_indices[np.argmax(tpr[valid_indices])]
#     train_tpr_at_fpr = tpr[best_index]
# probs = model.predict_proba(ARCHIVE_TEST_DF)[:,1]
# fpr, tpr, thresholds = roc_curve(y_test, probs)
# if np.any(fpr <= target_fpr):
#     valid_indices = np.where(fpr <= target_fpr)[0]
#     best_index = valid_indices[np.argmax(tpr[valid_indices])]
#     baseline_tpr_at_fpr = tpr[best_index]
# print(f"Test TPR: {baseline_tpr_at_fpr}, Test shape: {ARCHIVE_TEST_DF.shape}")
# # print(f"Train TPR: {train_tpr_at_fpr}, Train shape: {ARCHIVE_TRAIN_DF.shape}")

In [419]:
# #make a copy of the dataframe ARCHIVE_TRAIN_DF and ARCHIVE_TEST_DF
# ARCHIVE_TRAIN_DF_final = ARCHIVE_TRAIN_DF.copy()
# ARCHIVE_TEST_DF_final = ARCHIVE_TEST_DF.copy()

In [420]:


# def tpr_at_fpr(y_true, y_score, target_fpr=0.05):
#     fpr, tpr, thresholds = roc_curve(y_true, y_score)
#     if np.any(fpr <= target_fpr):
#         valid_indices = np.where(fpr <= target_fpr)[0]
#         best_index = valid_indices[np.argmax(tpr[valid_indices])]
#         return tpr[best_index]
#     else:
#         return 0.0

# def scorer(estimator, X, y):
#     probs = estimator.predict_proba(X)[:,1]
#     return tpr_at_fpr(y, probs, target_fpr=0.05)

# search = RandomizedSearchCV(
#     DecisionTreeClassifier(random_state=42),
#     param_distributions={
#         'max_depth': [4,6,8,10],
#         'min_samples_split': [2,5,10,20],
#         'min_samples_leaf': [1,2,4,8],
#         'max_features': ['sqrt', 'log2', None],
#         'criterion': ['gini', 'entropy']
#     },
#     n_iter=100,
#     scoring=scorer,
#     cv=2,
#     verbose=3,
# )

# search.fit(ARCHIVE_TRAIN_DF_final, y_train)

In [421]:
# best_tree = search.best_estimator_
# best_score = search.best_score_
# print(best_tree, best_score)